# **Day 34: Multi-class, Multi-label and Multi-output Learning**
*Module 5 - Classification*

Not every problem is "one of two classes".

| Problem | Each sample has | Example |
|---|---|---|
| **Binary** | one of 2 classes | landslide / no landslide |
| **Multi-class** | exactly one of K classes | land-cover class of a pixel |
| **Multi-label** | any subset of K labels | image tags: {cloud, river, road}; a scene with crops AND trees |
| **Multi-output regression** | several numbers | soil sand/silt/clay %, yield + protein |
| **Hierarchical** | a path in a class tree | vegetation > forest > deciduous |
| **Ordinal** | ordered classes | damage: none < minor < major < destroyed |

> Multi-class means choosing one of many classes; multi-label means a sample can have several labels at once; multi-output means predicting several targets together. Each needs the right strategy and metrics.
>
> *Example:* A satellite scene can be both "river" and "agriculture" at once. That is multi-label, not multi-class.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(34)

## **1. Multi-Class Strategies**
- **Natively multi-class**: trees, Random Forest, kNN, Naive Bayes, LDA/QDA, multinomial logistic regression, neural networks (softmax).
- **One-vs-Rest (OvR)**: K binary classifiers "class k vs all others"; predict the highest score.
- **One-vs-One (OvO)**: $K(K-1)/2$ classifiers, one per pair; majority vote. Each trains on less data (used by `SVC`).
- **Error-Correcting Output Codes (ECOC)**: each class gets a binary code word; robust to individual classifier errors.

In [2]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier, OutputCodeClassifier
from sklearn.datasets import load_digits
X, y = load_digits(return_X_y=True)
base = make_pipeline(StandardScaler(), LinearSVC(C=0.05, max_iter=10000))
strategies = {"OvR (10 classifiers)": OneVsRestClassifier(base), "OvO (45 classifiers)": OneVsOneClassifier(base),
              "ECOC (code size 2 -> 20 classifiers)": OutputCodeClassifier(base, code_size=2, random_state=0),
              "Multinomial logistic (native)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))}
for name, m in strategies.items():
    print(f"{name:<38} CV accuracy {cross_val_score(m, X, y, cv=5, n_jobs=-1).mean():.3f}")

OvR (10 classifiers)                   CV accuracy 0.917


OvO (45 classifiers)                   CV accuracy 0.947


ECOC (code size 2 -> 20 classifiers)   CV accuracy 0.858


Multinomial logistic (native)          CV accuracy 0.920


## **2. Multi-Label Classification**
Example: tagging satellite image tiles with several labels at once (like the BigEarthNet benchmark). We simulate tiles described by 12 features, with 5 labels that are **correlated** (e.g. "river" makes "irrigated crops" more likely).

In [3]:
n = 3000
Xml = rng.normal(size=(n, 12))
z_river = Xml[:, 0] + 0.5 * Xml[:, 1]
labels = pd.DataFrame({"river": z_river > 0.8})
labels["irrigated_crop"] = (0.9 * z_river + Xml[:, 2] + rng.normal(0, 0.7, n)) > 0.6
labels["forest"] = (Xml[:, 3] - 0.5 * Xml[:, 2] + rng.normal(0, 0.5, n)) > 0.5
labels["urban"] = (Xml[:, 4] + 0.3 * Xml[:, 5] - 0.4 * Xml[:, 3] + rng.normal(0, 0.5, n)) > 0.9
labels["cloud"] = (Xml[:, 6] + rng.normal(0, 0.3, n)) > 1.0
Y = labels.astype(int).values
print("Label frequencies:\n", labels.mean().round(3))
print("\nNumber of labels per tile:", np.bincount(Y.sum(1)))
print("\nLabel correlation:\n", labels.astype(int).corr().round(2))

Label frequencies:
 river             0.222
irrigated_crop    0.335
forest            0.326
urban             0.227
cloud             0.161
dtype: float64

Number of labels per tile: [ 704 1184  766  290   54    2]

Label correlation:
                 river  irrigated_crop  forest  urban  cloud
river            1.00            0.40    0.01  -0.00   0.02
irrigated_crop   0.40            1.00   -0.14  -0.01   0.02
forest           0.01           -0.14    1.00  -0.13  -0.01
urban           -0.00           -0.01   -0.13   1.00  -0.01
cloud            0.02            0.02   -0.01  -0.01   1.00


### Strategies
- **Binary relevance**: one independent classifier per label (`MultiOutputClassifier`). Simple; ignores label correlations.
- **Classifier chain**: classifier $k$ also receives the predictions for labels $1..k-1$ as features; exploits correlations. Order matters, so an **ensemble of random chains** is more robust.
- **Native multi-label**: Random Forest, kNN and neural networks (sigmoid outputs + binary cross-entropy) accept a 2-D `Y` directly.

In [4]:
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, jaccard_score
X_tr, X_te, Y_tr, Y_te = train_test_split(Xml, Y, test_size=0.3, random_state=0)
lr = LogisticRegression(max_iter=2000)
models = {"Binary relevance (LogReg)": MultiOutputClassifier(lr),
          "Classifier chain (LogReg)": ClassifierChain(lr, order=[0, 1, 2, 3, 4], random_state=0),
          "Ensemble of 10 random chains": None,
          "Random Forest (native)": RandomForestClassifier(300, n_jobs=-1, random_state=0)}

def ml_report(Y_true, Y_pred):
    return {"Hamming loss": hamming_loss(Y_true, Y_pred), "subset accuracy": accuracy_score(Y_true, Y_pred),
            "F1 micro": f1_score(Y_true, Y_pred, average="micro"), "F1 macro": f1_score(Y_true, Y_pred, average="macro"),
            "F1 samples": f1_score(Y_true, Y_pred, average="samples", zero_division=0), "Jaccard samples": jaccard_score(Y_true, Y_pred, average="samples", zero_division=0)}

rows = {}
for name, m in models.items():
    if m is None:
        chains = [ClassifierChain(lr, order="random", random_state=i).fit(X_tr, Y_tr) for i in range(10)]
        P = np.mean([c.predict_proba(X_te) for c in chains], axis=0); Y_pred = (P >= 0.5).astype(int)
    else:
        Y_pred = m.fit(X_tr, Y_tr).predict(X_te)
    rows[name] = ml_report(Y_te, Y_pred)
pd.DataFrame(rows).T.round(3)

,Hamming loss,subset accuracy,F1 micro,F1 macro,F1 samples,Jaccard samples
Binary relevance (LogReg),0.090,0.611,0.820,0.822,0.609,0.576
Classifier chain (LogReg),0.089,0.613,0.822,0.823,0.611,0.578
Ensemble of 10 random chains,0.090,0.611,0.819,0.821,0.608,0.574
Random Forest (native),0.109,0.553,0.771,0.773,0.562,0.524


| Metric | Meaning | Strictness |
|---|---|---|
| Hamming loss | Fraction of wrong individual labels | Lenient (lower = better) |
| Subset accuracy | All labels of a sample exactly right | Very strict |
| F1 micro | Pools all label decisions | Dominated by frequent labels |
| F1 macro | Mean F1 per label | Rare labels count fully |
| F1 / Jaccard samples | Per-sample overlap of predicted and true sets | Intuitive per tile |

Per-label thresholds can be tuned separately (Day 29) because labels have different frequencies and costs.

## **3. Multi-Output Regression**
Predict several continuous targets at once, e.g. soil texture fractions (sand, silt, clay) from spectra and terrain.

In [5]:
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
Xs = rng.normal(size=(1500, 10))
sand = 40 + 8 * Xs[:, 0] - 4 * Xs[:, 1] + rng.normal(0, 3, 1500)
clay = 25 - 5 * Xs[:, 0] + 3 * Xs[:, 2] + rng.normal(0, 3, 1500)
silt = 100 - sand - clay                                      # compositional: sums to 100
Ys = np.c_[sand, silt, clay]
Xs_tr, Xs_te, Ys_tr, Ys_te = train_test_split(Xs, Ys, random_state=0)
for name, m in {"Separate Ridge models": MultiOutputRegressor(RidgeCV()), "Regressor chain": RegressorChain(RidgeCV(), order=[0, 2, 1]),
                "Random Forest (native)": RandomForestRegressor(300, n_jobs=-1, random_state=0)}.items():
    P = m.fit(Xs_tr, Ys_tr).predict(Xs_te)
    print(f"{name:<24} R2 sand/silt/clay = {np.round(r2_score(Ys_te, P, multioutput='raw_values'), 3)} | mean |sum - 100| = {np.abs(P.sum(1) - 100).mean():.2f}")

Separate Ridge models    R2 sand/silt/clay = [0.895 0.627 0.789] | mean |sum - 100| = 0.00
Regressor chain          R2 sand/silt/clay = [0.895 0.627 0.789] | mean |sum - 100| = 0.00


Random Forest (native)   R2 sand/silt/clay = [0.862 0.575 0.742] | mean |sum - 100| = 0.00


Here every model's predictions happen to sum to 100 (linear models, and averages of training compositions, preserve sums). But nothing stops a linear model from predicting **negative** fractions or values above 100% for unusual inputs. For **compositional** targets, model log-ratios (e.g. the additive log-ratio transform, Practice Problem 4): back-transformed predictions are then always positive and sum to 100%.

# **Part 2: Going Deeper**

### Ordinal classification
Building-damage grades (none < minor < major < destroyed) are ordered. Standard multi-class ignores the order: predicting "destroyed" for a "none" building costs the same as predicting "minor". **Frank & Hall (2001)**: train K-1 binary classifiers for "$y > k$" and combine $P(y=k) = P(y>k-1) - P(y>k)$.

In [6]:
class OrdinalClassifier:
    def __init__(self, base):
        self.base = base
    def fit(self, X, y):
        from sklearn.base import clone
        self.classes_ = np.sort(np.unique(y))
        self.models_ = [clone(self.base).fit(X, (y > k).astype(int)) for k in self.classes_[:-1]]
        return self
    def predict_proba(self, X):
        gt = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models_])          # P(y > k)
        gt = np.minimum.accumulate(gt, axis=1)                                             # enforce monotonicity
        P = np.column_stack([1 - gt[:, 0]] + [gt[:, i - 1] - gt[:, i] for i in range(1, gt.shape[1])] + [gt[:, -1]])
        return P.clip(0)
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

latent = Xml[:, 0] * 1.2 + Xml[:, 1] + rng.normal(0, 0.8, n)
damage = np.digitize(latent, [-1.0, 0.3, 1.5])                         # 0..3
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(Xml, damage, test_size=0.3, random_state=0)
from sklearn.metrics import mean_absolute_error
for name, m in {"nominal multi-class": LogisticRegression(max_iter=2000), "ordinal (Frank & Hall)": OrdinalClassifier(LogisticRegression(max_iter=2000))}.items():
    pr = m.fit(Xo_tr, yo_tr).predict(Xo_te)
    print(f"{name:<24} accuracy {np.mean(pr == yo_te):.3f} | mean absolute grade error {mean_absolute_error(yo_te, pr):.3f}")

nominal multi-class      accuracy 0.624 | mean absolute grade error 0.401
ordinal (Frank & Hall)   accuracy 0.623 | mean absolute grade error 0.397


Here the gain is small: logistic regression already orders the classes along one latent direction. Ordinal decompositions help more with flexible base models (trees, boosting) and when misclassifying by several grades is costly. Always report an **ordinal-aware metric** (mean absolute grade error, quadratic-weighted kappa) alongside accuracy.

### Hierarchical classification
Classes often form a tree: Vegetation -> {Forest -> {Deciduous, Evergreen}, Cropland}. A **local classifier per parent node** first predicts the coarse class, then the fine class within it. Benefits: errors stay "close" in the hierarchy, and we can report the coarse level when the fine level is uncertain. Hierarchical metrics give partial credit for correct ancestors.

## **Practice Problems**
1. Tune one threshold per label for the binary-relevance model to maximise each label's F1 on a validation split. Does macro F1 improve?
2. Which multi-label strategy has the best subset accuracy, and why might chains help that metric most?
3. Compute the confusion matrix of the ordinal model. Are its errors mostly between neighbouring grades?
4. *(Advanced)* Fit a model on additive log-ratios (ALR) of sand/silt/clay and back-transform. Check that predictions sum exactly to 100.

In [7]:
# Solution 1
X_a, X_v, Y_a, Y_v = train_test_split(X_tr, Y_tr, test_size=0.3, random_state=1)
br = MultiOutputClassifier(lr).fit(X_a, Y_a)
Pv = np.column_stack([e.predict_proba(X_v)[:, 1] for e in br.estimators_])
ths = [max(np.linspace(0.05, 0.95, 91), key=lambda t: f1_score(Y_v[:, k], Pv[:, k] >= t)) for k in range(Y.shape[1])]
Pt = np.column_stack([e.predict_proba(X_te)[:, 1] for e in br.estimators_])
print("thresholds:", np.round(ths, 2))
print("macro F1 at 0.5:", round(f1_score(Y_te, (Pt >= 0.5).astype(int), average="macro"), 3), "| tuned:", round(f1_score(Y_te, (Pt >= np.array(ths)).astype(int), average="macro"), 3))
# Solution 3
from sklearn.metrics import confusion_matrix
print(confusion_matrix(yo_te, OrdinalClassifier(LogisticRegression(max_iter=2000)).fit(Xo_tr, yo_tr).predict(Xo_te)))
# Solution 4
def alr_forward(Y):  return np.c_[np.log(Y[:, 0] / Y[:, 1]), np.log(Y[:, 2] / Y[:, 1])]
def alr_inverse(Z):
    e = np.c_[np.exp(Z[:, 0]), np.ones(len(Z)), np.exp(Z[:, 1])]; return 100 * e / e.sum(1, keepdims=True)
Ys_pos = Ys.clip(1)
Ys_pos = 100 * Ys_pos / Ys_pos.sum(1, keepdims=True)
_, _, Yp_tr, Yp_te = train_test_split(Xs, Ys_pos, random_state=0)          # same split as before (same random_state)
alr_model = MultiOutputRegressor(RidgeCV()).fit(Xs_tr, alr_forward(Yp_tr))
P_alr = alr_inverse(alr_model.predict(Xs_te))
print("ALR model: all sums = 100:", np.allclose(P_alr.sum(1), 100), "| min fraction:", P_alr.min().round(2),
      "| R2:", np.round(r2_score(Yp_te, P_alr, multioutput="raw_values"), 3))

thresholds: [0.46 0.4  0.42 0.39 0.41]
macro F1 at 0.5: 0.823 | tuned: 0.828
[[173  73   2   0]
 [ 44 157  58   3]
 [  5  64 117  36]
 [  0   8  46 114]]
ALR model: all sums = 100: True | min fraction: 7.5 | R2: [0.894 0.615 0.769]


## **Key References**
- Allwein, E. L., Schapire, R. E., & Singer, Y. (2000). Reducing multiclass to binary: A unifying approach for margin classifiers. *JMLR*, 1, 113-141.
- Read, J., Pfahringer, B., Holmes, G., & Frank, E. (2011). Classifier chains for multi-label classification. *Machine Learning*, 85(3), 333-359.
- Tsoumakas, G., & Katakis, I. (2007). Multi-label classification: An overview. *International Journal of Data Warehousing and Mining*, 3(3), 1-13.
- Frank, E., & Hall, M. (2001). A simple approach to ordinal classification. *ECML 2001*, 145-156.
- Silla, C. N., & Freitas, A. A. (2011). A survey of hierarchical classification across different application domains. *Data Mining and Knowledge Discovery*, 22(1-2), 31-72.
- Sumbul, G. et al. (2019). BigEarthNet: A large-scale benchmark archive for remote sensing image understanding. *IGARSS 2019*.
- Aitchison, J. (1986). *The Statistical Analysis of Compositional Data*. Chapman & Hall.